Capstone and Frontiers
======================

**Author:** Ethan Ligon



Where the methods break down, and then your own analysis.



## Reading



### Reading



## Two measures of poverty, and they disagree



## Full risk sharing



## Why covariate shocks break the test



## The sign flip, computed



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

# These notebooks are read two ways: on your own screen, and projected onto
# a wall in a lit room where thin lines and pale markers are simply not
# there.  Those want different plots, so the difference is a switch rather
# than a compromise that suits neither.  Leave it False; the lecturer flips
# it.  Everything below the `if' is about the wall, not about your laptop.
PROJECTOR = False

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})
if PROJECTOR:
    mpl.rcParams.update({
        'figure.figsize': (7, 4.5),
        'font.size': 15, 'axes.labelsize': 15, 'axes.titlesize': 17,
        'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 13,
        'lines.linewidth': 3.0, 'lines.markersize': 9, 'axes.linewidth': 1.5,
    })

from pathlib import Path
import lsms_library as ll
import cfe
from cfe import Regression
import numpy as np, pandas as pd

def uganda_cfe():
    # The estimated CFE system for Uganda, as a cfe.Regression.  The object
    # carries beta, gamma, w = -log lambda, and predicted expenditures, so
    # nothing downstream has to refit.  Uses a copy staged on the hub if
    # there is one, else a copy you estimated earlier, else estimates it
    # from scratch (about forty seconds) and caches the result.  So this
    # cell is self-contained: no other notebook need have been run first.
    staged = Path('/srv/data/hhsurveys/uganda.rgsn')
    mine   = Path.home() / '.cache' / 'hhsurveys' / 'uganda.rgsn'
    for c in (staged, mine):
        if c.exists():
            return cfe.read_pickle(str(c))

    uga = ll.Country('Uganda')
    x = uga.food_expenditures().squeeze()
    agg = (uga.categorical_mapping['harmonize_food']
              .set_index('Preferred Label')['Aggregate Label'].to_dict())
    x = x.rename(index=agg, level='j')
    x = x.groupby(x.index.names).sum()

    y = np.log(x.replace(0, np.nan).dropna()).groupby(['i', 't', 'j']).sum()
    y = pd.concat({1: y}, names=['m']).reorder_levels(['i', 't', 'm', 'j']).sort_index()

    d0 = uga.household_characteristics()
    d = d0.assign(
        Girls=d0[[f'F {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Boys =d0[[f'M {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Women=d0[[f'F {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
        Men  =d0[[f'M {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
    )[['Girls', 'Boys', 'Women', 'Men', 'log HSize']].dropna(how='any')
    d = d.groupby(['i', 't']).first()
    d = pd.concat({1: d}, names=['m']).reorder_levels(['i', 't', 'm']).sort_index()

    r = Regression(y=y, d=d)
    r.get_beta(); r.get_w(); r.predicted_expenditures()
    mine.parent.mkdir(parents=True, exist_ok=True)
    r.to_pickle(str(mine))
    return r

r = uganda_cfe()
w = r.get_w()

waves = sorted(w.index.get_level_values('t').unique())
base = waves[0]

# Anchor at 58% in the base wave, then carry the SAME welfare threshold
# forward.  This is the whole exercise: one line, one threshold, no index.
z = w.xs(base, level='t').quantile(0.58)
hc = w.groupby('t').apply(lambda s: float((s <= z).mean()))
print(f"headcount on w, anchored at 58% in {base}:")
print(hc.round(3).to_string())

In [1]:
# The conventional measure, undeflated: nominal per-capita expenditure.
uga = ll.Country('Uganda')
xt = uga.food_expenditures().squeeze().groupby(['i', 't']).sum()
n = np.exp(uga.household_characteristics()['log HSize']).groupby(['i', 't']).first()
pc = (xt / n.reindex(xt.index)).dropna()

med = pc.groupby('t').median()
print("median nominal per-capita food expenditure:")
print(med.round(0).to_string())
print(f"\nchange {waves[0]} -> {waves[1]}: "
      f"{100 * (med.loc[waves[1]] / med.loc[waves[0]] - 1):+.0f}%")
print(f"headcount on w, same interval: "
      f"{hc.loc[waves[0]]:.3f} -> {hc.loc[waves[1]]:.3f}")

One measure says welfare fell; the other says spending rose by a third.
Reconciling them requires a deflator larger than the measured inflation over
the period, which is to say: the conventional answer depends entirely on a
number the survey does not contain.



## A risk-sharing test



In [1]:
import statsmodels.api as sm

sh = uga.shocks()
print(sh.index.get_level_values('Shock').unique().tolist())

In [1]:
# One indicator per shock type, per household-wave.
S = (sh.assign(hit=1.0).hit
       .groupby(['i', 't', 'Shock']).max()
       .unstack('Shock').fillna(0.0))

wi = w.droplevel('m') if 'm' in (w.index.names or []) else w
df = pd.concat([wi.rename('w'), S], axis=1).dropna(subset=['w'])
df[S.columns] = df[S.columns].fillna(0.0)

# Keep the shocks that are common enough to estimate.
keep = [c for c in S.columns if df[c].mean() > 0.02]
print(f"{len(keep)} shock types with >2% incidence")

D = pd.get_dummies(df.index.get_level_values('t'), prefix='t',
                   drop_first=True, dtype=float)
D.index = df.index
X = pd.concat([df[keep], D], axis=1)
Z = pd.concat([df.w, X], axis=1)
Zd = Z - Z.groupby(level='i').transform('mean')     # household effects

res = sm.OLS(Zd.w, Zd.drop(columns='w')).fit(
    cov_type='cluster', cov_kwds={'groups': Zd.index.get_level_values('i')})
print(res.summary().tables[1])

Read the shock coefficients, not the time dummies.  Under full risk sharing
every one of them is zero.  Where they are not, that household bore the
shock itself.

And then read what is missing.  A drought that struck the whole country in a
given wave contributes nothing to these estimates: the time dummy took it.
Whatever this regression says about insurance, it says it only about the
shocks that hit some households and not others in the same period.



## Three more places it breaks



## The capstone



## What counts as reproducible



## Setting up your capstone



In [1]:
import lsms_library as ll

# Everything the library knows about
ll.countries()

In [1]:
# Which tables exist where?  `coverage` is a country-by-table matrix.
# refresh='coverage' reads the declarations directly, so it does not need a
# prebuilt snapshot on disk.
cov = ll.coverage(refresh='coverage')
cov.head(20)

In [1]:
# Pick one and look before you leap
c = ll.Country('Uganda')       # <- change me
print(c.waves)
print(c.data_scheme)

## A reproducibility check



Put this at the top of the notebook you present.



In [1]:
import sys, platform
from importlib.metadata import version
import lsms_library, pandas, numpy, statsmodels

print(f"python       {sys.version.split()[0]}  ({platform.system()})")
print(f"lsms_library {lsms_library.__version__}")
print(f"CFEDemands   {version('CFEDemands')}")
print(f"pandas       {pandas.__version__}")
print(f"numpy        {numpy.__version__}")
print(f"statsmodels  {statsmodels.__version__}")

If you have edited a country's configuration or scripts, the parquet cache
will be stale, and there is no automatic staleness check:



In [1]:
# Force a rebuild for one country (slow; only when you have changed a config)
# !LSMS_NO_CACHE=1 python -c "import lsms_library as ll; ll.Country('Uganda').sample()"

## Exercises



The capstone is the exercise.  Two suggestions if you finish early:

1.  Take your estimate from (3) and compute it a second way — a different
    aggregate, a different scale, a different clustering level.  Report the
    range.  That range, not the standard error, is your real uncertainty.
2.  Hand your notebook to the person next to you and have them run it from a
    clean kernel.  Fix whatever breaks.  That is the exercise.

